In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

df = pd.read_csv("winequalityN.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print()
df.info()
print()
print("Nulls per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print(df["type"].value_counts())

In [ ]:
print(df["quality"].value_counts().sort_index())

plt.figure(figsize=(7,4))
sns.countplot(x="quality", data=df, color="steelblue")
plt.title("Distribution of Wine Quality Scores")
plt.show()

## Class Imbalance

About 93% of the wines have a quality score of 5, 6 or 7. Scores 3, 8 and 9 are very rare (30, 149 and 5 wines). This causes two problems: a model can reach a decent accuracy by predicting only the common scores, and it has too few examples to learn the rare ones. For this reason I use a stratified train/test split, and I look at precision, recall and the confusion matrix for each class and not just accuracy.

In [ ]:
print("Before:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", df.shape)

num_cols = df.select_dtypes("number").columns.drop("quality")
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
print("Nulls left:", df.isnull().sum().sum())

df["type"] = df["type"].map({"white": 0, "red": 1})
print(df["quality"].value_counts().sort_index())

## Data Cleaning

The dataset has 6,497 wines (4,898 white and 1,599 red) and 13 columns. 38 values were missing in 7 columns (under 0.6% of the data), so I filled them with each column's median, which is not affected by outliers.

I also found 1,168 duplicate rows (18%) and removed them. Duplicates are a problem in machine learning: an identical wine can end up in both the training and test sets, so the model is tested on data it has already seen and the scores look better than they really are. After cleaning, 5,329 wines remain. The wine `type` was encoded as 0 (white) and 1 (red).

In [ ]:
features = df.columns.drop(["quality", "type"])
df[features].hist(bins=30, figsize=(14,10), color="steelblue")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11,8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()

print(df.corr()["quality"].sort_values(ascending=False).round(2))

## Feature Distributions and Correlations

Most chemical features are right-skewed (for example residual sugar, chlorides and volatile acidity), with a few extreme values. pH is roughly symmetric.

Alcohol has the strongest positive correlation with quality (0.47), and density (-0.33) and volatile acidity (-0.26) have the strongest negative ones. No feature is very strongly correlated with quality, so predicting it will not be easy.

Some features overlap strongly: density and alcohol (-0.67), and free and total sulfur dioxide (0.72). Part of density's link to quality is probably the alcohol effect.

In [ ]:
def bin_quality(q):
    if q <= 5:
        return "low"
    elif q == 6:
        return "medium"
    return "high"

df["quality_group"] = df["quality"].apply(bin_quality)
print(df["quality_group"].value_counts())
print((df["quality_group"].value_counts(normalize=True) * 100).round(1))

## Quality Binning

The original quality scores run from 3 to 9, but scores 3, 8 and 9 have only 30, 149 and 5 wines. A 7-class model would have too few examples for the rare classes, so I grouped the scores into three classes:
- low: 3, 4, 5 (1,991 wines, 37.4%)
- medium: 6 (2,327 wines, 43.7%)
- high: 7, 8, 9 (1,011 wines, 19.0%)

Every class now has enough examples, and the groups have a clear meaning (below average, average, above average). The "high" class is still the smallest.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["quality", "quality_group"])
y = df["quality_group"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(y_train.value_counts(normalize=True).round(3))
print(y_test.value_counts(normalize=True).round(3))

## Train/Test Split and Scaling

The data was split 80/20 (4,263 training and 1,066 test wines) with stratification, so each class has the same percentage in both sets (about 37% low, 44% medium, 19% high). I standardised the features with StandardScaler, fitted on the training data only so that no information from the test set leaks in. SVC and SGD are sensitive to feature scale, so they use the scaled data. Random Forest does not need scaling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf  = RandomForestClassifier(n_estimators=200, random_state=42)
sgd = SGDClassifier(random_state=42)
svc = SVC(random_state=42)

rf.fit(X_train, y_train)          # trees don't need scaling
sgd.fit(X_train_s, y_train)
svc.fit(X_train_s, y_train)

preds = {
    "Random Forest": rf.predict(X_test),
    "SGD": sgd.predict(X_test_s),
    "SVC": svc.predict(X_test_s),
}

labels = ["low", "medium", "high"]
for name, p in preds.items():
    print("=" * 50)
    print(name, "| accuracy:", round(accuracy_score(y_test, p), 3))
    print(classification_report(y_test, p, labels=labels))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16,4))
for a, (name, p) in zip(ax, preds.items()):
    cm = confusion_matrix(y_test, p, labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=a, cbar=False)
    a.set_title(name)
    a.set_xlabel("Predicted")
    a.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## Model Evaluation

| Model | Accuracy | Macro F1 |
|---|---|---|
| Random Forest | 0.62 | 0.61 |
| SVC | 0.592 | 0.57 |
| SGD | 0.536 | 0.52 |

Random Forest performed best overall and was the best at finding high-quality wines (recall 0.51). SVC rarely predicted "high" (recall 0.37), although it was fairly precise when it did (0.66). SGD, a linear model, performed worst.

In the confusion matrices, almost all mistakes are between neighbouring classes (low and medium, or medium and high), and very few wines were mistaken by two classes. This is expected because quality is an ordered scale and the chemistry of wines scored 5, 6 and 7 is similar. All models find the "high" class hardest, partly because it has the fewest examples.

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(8,5))
imp.plot(kind="barh", color="steelblue")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print(imp.sort_values(ascending=False).round(3))

## Feature Importance

Alcohol is the most important feature in the Random Forest (0.139), followed by density (0.102) and volatile acidity (0.100). These are the same features that had the strongest correlations with quality, so the model agrees with the earlier analysis. Importances for the middle features are close together (about 0.07 to 0.09), so their order should not be over-interpreted. Wine type has almost no importance (0.004), probably because the chemical measurements already contain the difference between red and white wines. Feature importance shows how much the model uses a feature, not whether it raises or lowers quality, so I use the correlations for the direction.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

rows = []
for name, p in preds.items():
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, p, average="macro", zero_division=0)
    rows.append([name, accuracy_score(y_test, p), prec, rec, f1])

comparison = pd.DataFrame(rows, columns=["Model", "Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]).round(3)
comparison

## Model Comparison

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1 |
|---|---|---|---|---|
| Random Forest | 0.620 | 0.627 | 0.602 | 0.612 |
| SVC | 0.592 | 0.615 | 0.553 | 0.569 |
| SGD | 0.536 | 0.522 | 0.529 | 0.525 |

Random Forest is the best on every metric. SVC is second, and SGD is last.

## Conclusion

Random Forest is the most suitable model of the three. It has the highest accuracy (0.62) and macro F1 (0.61), and it is the best at finding high-quality wines (recall 0.51). It also needs no feature scaling and gives feature importances, which helps explain its predictions. SVC is a reasonable second choice but misses many high-quality wines (recall 0.37). SGD, a linear model, is too simple for this data.

Still, 0.62 accuracy is only moderate (always predicting "medium" would give about 44%), so I would not use the model alone to decide a wine's quality in production.

**Limitations:**
- Sensory quality scores are subjective, and similar wines get different scores, which limits any model.
- The "high" class has only 19% of the wines, and all models struggle with it.
- I used default settings and did not tune hyperparameters.
- Red and white wines were modelled together.

**Possible improvements:** hyperparameter tuning, class weights or oversampling for the "high" class, separate models for red and white wine, and gradient boosting.